In [1]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [2]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [3]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [4]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [7]:
import sys
sys.path.append("..")

In [8]:
from dotenv import load_dotenv
from openai import OpenAI
from module4.evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [9]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [10]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [11]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

100%|██████████| 515/515 [02:10<00:00,  3.95it/s]


In [12]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [13]:
df_eval = pd.DataFrame(evaluations)

In [14]:
df_eval.head()

,question,document,score,reasoning
0,Is it still okay to join this course if I foun...,74eb249bbf,good,The AI answer preserves the key meaning of the...
1,Can I enroll after the course has already star...,74eb249bbf,bad,The AI answer correctly says enrollment after ...
2,"If I join the course now, can I still get a ce...",74eb249bbf,good,The AI answer preserves the core meaning: join...
3,Do I need to finish the project before submiss...,74eb249bbf,bad,The ground truth says yes: to receive a certif...
4,What’s the deadline for submitting the project...,74eb249bbf,good,The AI answer preserves the key point from the...


In [15]:
calc_total_price(usages)

0.3554264999999999

In [16]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 503/515 = 97.67%


In [18]:
pd.set_option("display.max_colwidth", None)

In [19]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
1,Can I enroll after the course has already started?,74eb249bbf,bad,"The AI answer correctly says enrollment after the course has started is allowed. However, it misses the key condition from the ground truth: to receive a certificate, the project must be submitted while submissions are still being accepted. Instead, it adds unrelated details about videos, GitHub materials, and deadlines. Because the important certificate condition is omitted, the answer is not fully equivalent."
3,Do I need to finish the project before submissions close to get certified?,74eb249bbf,bad,"The ground truth says yes: to receive a certificate, the project must be submitted while submissions are still open. The AI answer fails to convey this and instead says it does not know, which is incorrect and misses the key point."
40,When is the next run of the course scheduled for?,bd31146b0e,bad,"The ground truth states a specific scheduled time: Summer 2027. The AI answer says it does not know, which fails to provide the correct information and is not semantically equivalent."
103,Which model should I pick in `responses.create(...)` if `gpt-4o` is not available to me?,152af39a53,bad,"The AI answer is not semantically equivalent to the ground truth. The ground truth explains that the issue is an API quota/billing problem and instructs the user to add credits, possibly recreate the API key, and then choose one of the models available to their account in `responses.create(...)`. The AI answer instead recommends a specific pinned model (`openai/gpt-oss-120b:free`) and discusses OpenRouter routing, which is unrelated and does not address the quota/billing resolution or the generic guidance to pick an available model. It misses the key point and appears to answer a different question."
224,What is the right way to set up the get_weather tool so Azure OpenAI accepts it?,b9c7c6bb34,bad,"The AI answer does not provide the required setup for the `get_weather_tool` JSON and explicitly says it doesn't know. It misses all key information from the ground truth, which specifies the exact tool schema Azure OpenAI accepts."


In [22]:
df_answers[df_answers["document"]=="74eb249bbf"]

,question,answer_llm,answer_orig,document
0,Is it still okay to join this course if I found it late?,"Yes, you can still join the course even if you found it late. If you want a certificate, make sure you submit your project while submissions are still being accepted.","Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.",74eb249bbf
1,Can I enroll after the course has already started?,"You can start whenever you want, so yes, you can enroll after the course has already started. The videos and GitHub materials are available, and the deadlines are listed on the course management platform.","Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.",74eb249bbf
2,"If I join the course now, can I still get a certificate?","Yes, but only if you join the live cohort and submit your project while submissions are still open. Certificates are not available for self-paced mode.","Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.",74eb249bbf
3,Do I need to finish the project before submissions close to get certified?,I don't know.,"Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.",74eb249bbf
4,What’s the deadline for submitting the project if I want the certificate?,"If you want a certificate, you need to submit your project while submissions are still open. There isn’t a specific deadline given here, just that you must submit before the project submission form closes.","Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.",74eb249bbf
